In [0]:
# Celda 1: instalar optbinning (reinicia Python, por eso va primero)
%pip install optbinning

In [0]:
# reiniciar phyton por instalación de optbinning
dbutils.library.restartPython()

In [0]:
# Celda 2: base de modelación con las transformaciones
from pyspark.sql import functions as F

inv = spark.table("workspace.default.lc_variable_inventory").toPandas()
cand = inv.loc[inv["decision"] == "Candidata", "variable"].tolist()
cand = [c for c in cand if c not in ("funded_amnt", "funded_amnt_inv", "fico_range_high",
                                     "earliest_cr_line", "emp_length")]
cand = [c for c in cand if c not in ("funded_amnt", "funded_amnt_inv", "fico_range_high",
                                     "earliest_cr_line", "emp_length", "addr_state")]
cand += ["credit_hist_months", "emp_length_yrs"]

mod = (spark.table("workspace.default.lc_population")
       # Antigüedad crediticia en meses al momento de originar
       .withColumn("credit_hist_months",
                   F.months_between("issue_date", F.to_date("earliest_cr_line", "MMM-yyyy")).cast("int"))
       # "< 1 year" -> 0, "10+ years" -> 10, "n/a" -> nulo
       .withColumn("emp_length_yrs",
                   F.when(F.col("emp_length").startswith("<"), 0)
                    .otherwise(F.regexp_extract("emp_length", r"(\d+)", 1).cast("int")))
       .select("id", "issue_year", "sample", "target", *cand))

mod.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.lc_model_base")
print(f"Variables candidatas: {len(cand)}")


In [0]:
# Celda 3: binning óptimo e IV sobre DEV
from optbinning import BinningProcess

dev_pd = spark.table("workspace.default.lc_model_base").filter("sample = 'DEV'").toPandas()
X, y = dev_pd[cand], dev_pd["target"].values
categoricas = [c for c in cand if X[c].dtype == "object"]   # purpose, home_ownership, etc.

bp = BinningProcess(variable_names=cand, categorical_variables=categoricas)
bp.fit(X, y)

resumen = bp.summary().sort_values("iv", ascending=False)
display(resumen[["name", "dtype", "n_bins", "iv", "gini"]])


In [0]:
print([c for c in cand if c not in resumen["name"].values])
print("credit_hist_months" in dev_pd.columns, "emp_length_yrs" in dev_pd.columns)
